In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/SilverLoader"

In [0]:
silver_table_name = get_table_name("customer", "silver")
bronze_table_name = get_table_name("customer", "bronze")

In [0]:
from pyspark.sql import functions as F

# --- Transform function for silver customers ---
# Must carry _bronze_ingestion_ts through (required by SilverLoader for watermark tracking)
def transform_customers(bronze_df):
    silver_df = (bronze_df
        .select(
            F.col("customer_id"),
            F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")).alias("customer_name"),
            clean_text("email").alias("email"),
            F.col("phone"),
            clean_text("city").alias("city"),
            clean_text("state").alias("state"),
            clean_text("country").alias("country"),
            clean_text("customer_segment").alias("customer_segment"),
            F.col("signup_date"),
            F.col("updated_at").alias("customer_updated_at"),
            F.col("_bronze_ingestion_ts")
        )
    )
    # Validate keys — drop rows with null/empty/non-positive customer_id
    silver_df = validate_keys(silver_df, ["customer_id"])
    # Remove duplicates by customer_id, keeping the latest updated record
    silver_df = remove_duplicates(silver_df, ["customer_id"], order_col="customer_updated_at")
    # Drop any remaining bronze metadata columns
    silver_df = drop_bronze_metadata(silver_df)
    return silver_df

# --- Run incremental load (full on first run, MERGE on subsequent runs) ---
result = incremental_silver_load(
    bronze_table_name=bronze_table_name,
    silver_table_name=silver_table_name,
    key_columns=["customer_id"],
    transform_fn=transform_customers,
    update_condition="source.customer_updated_at >= target.customer_updated_at"
)

print(result)

In [0]:
display(spark.read.table(silver_table_name))

## Silver Customers — Incremental Load

This notebook incrementally loads customer data from bronze to silver:
- **First run**: full load with DQ (clean text, validate keys, dedup by `customer_id`)
- **Subsequent runs**: only new bronze rows (detected via `_ingestion_timestamp` watermark) are merged into the silver table
- Uses `incremental_silver_load()` from the common **SilverLoader** notebook
- Upsert condition: only updates existing rows when `source.customer_updated_at >= target.customer_updated_at`
